In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from xgboost import XGBRegressor
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
import requests
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
#pd.set_option("display.max_columns", None)
plt.rcParams["figure.dpi"] = 110
sns.set_theme(style="whitegrid", palette="muted")

In [3]:
DATA_DIR = Path("Data")
END_DATE = "2026-01-31"
DELAY_THRESHOLD = 5 

In [4]:
# notice that the 2022-2024 data are xlsx files while 2025+ is a csv file 
xlsx_sources = [
    (DATA_DIR/"ttc-subway-delay-data-2022.xlsx", "2022"),
    (DATA_DIR/"ttc-subway-delay-data-2023.xlsx", "2023"),
    (DATA_DIR/"ttc-subway-delay-data-2024.xlsx", "Subway"),
]
dfs = [pd.read_excel(p, sheet_name=s) for p, s in xlsx_sources]

# 2025+ csv has an extra _id column so we must drop it
df_2025 = pd.read_csv(DATA_DIR / "TTC Subway Delay Data since 2025.csv")
df_2025 = df_2025.drop(columns=["_id"], errors="ignore")
dfs.append(df_2025)

df = pd.concat(dfs, ignore_index=True)
print(f"Combined shape : {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Date range : {pd.to_datetime(df['Date']).min().date()} to {pd.to_datetime(df['Date']).max().date()}")
df.head(10)

Combined shape : 97,502 rows x 10 columns
Date range : 2022-01-01 to 2026-01-31


,Date,Time,Day,Station,Code,Min Delay,Min Gap,Bound,Line,Vehicle
0,2022-01-01 00:00:00,15:59,Saturday,LAWRENCE EAST STATION,SRDP,0,0,N,SRT,3023
1,2022-01-01 00:00:00,02:23,Saturday,SPADINA BD STATION,MUIS,0,0,NaN,BD,0
2,2022-01-01 00:00:00,22:00,Saturday,KENNEDY SRT STATION TO,MRO,0,0,NaN,SRT,0
3,2022-01-01 00:00:00,02:28,Saturday,VAUGHAN MC STATION,MUIS,0,0,NaN,YU,0
4,2022-01-01 00:00:00,02:34,Saturday,EGLINTON STATION,MUATC,0,0,S,YU,5981
5,2022-01-01 00:00:00,05:40,Saturday,QUEEN STATION,MUNCA,0,0,NaN,YU,0
6,2022-01-01 00:00:00,06:56,Saturday,DAVISVILLE STATION,MUNCA,0,0,NaN,YU,0
7,2022-01-01 00:00:00,06:58,Saturday,ST PATRICK STATION,MUNCA,0,0,NaN,YU,0
8,2022-01-01 00:00:00,07:01,Saturday,PAPE STATION,MUNCA,0,0,NaN,BD,0
9,2022-01-01 00:00:00,07:43,Saturday,WILSON STATION,TUATC,10,0,S,YU,5896


In [5]:
info_df = pd.DataFrame({
    "dtype"     : df.dtypes,
    "non_null"  : df.notna().sum(),
    "null_count": df.isna().sum(),
    "null_%"    : (df.isna().mean() * 100).round(1),
})
print(info_df.to_string())
print("Min Delay Statistics:")
print(df["Min Delay"].describe().round(2).to_string())

            dtype  non_null  null_count  null_%
Date       object     97502           0     0.0
Time          str     97502           0     0.0
Day           str     97502           0     0.0
Station       str     97502           0     0.0
Code          str     97502           0     0.0
Min Delay   int64     97502           0     0.0
Min Gap     int64     97502           0     0.0
Bound         str     63680       33822    34.7
Line          str     97296         206     0.2
Vehicle     int64     97502           0     0.0
Min Delay Statistics:
count    97502.00
mean         3.06
std         11.90
min          0.00
25%          0.00
50%          0.00
75%          4.00
max        900.00


In [7]:
# since Line 3 was discontinued, it is safe to drop it
length_before_drop = len(df)
df = df[df["Line"] != "SRT"].copy()
print(f"Rows dropped: {length_before_drop - len(df)}")  

# normalize line column since there are many unecessary varaints of the same line
LINE_NORM = {
    # YU (line 1) variants
    "YUS": "YU", "LINE 1": "YU",
    # BD (line 2) variants
    "B/D": "BD", "BD LINE 2": "BD", "BLOOR DANFORTH": "BD", "LINE 2 SHUTTLE": "BD",
    # SHP (line 4) variants
    "SHEP": "SHP",
    # Multi-Line variants
    "YUS/BD": "YU/BD",  "YU / BD": "YU/BD",  "YU/ BD": "YU/BD",
    "BD/YU":  "YU/BD",  "YU & BD": "YU/BD",  "YU/BD LINES": "YU/BD",
    "BD/YUS": "YU/BD",  "YU/BD LINE": "YU/BD", "YUS / BD": "YU/BD",
    "Y/BD":   "YU/BD",  "YUS AND BD": "YU/BD", "YUS/ BD": "YU/BD",
    "BD/ YU": "YU/BD",  "BD / YU": "YU/BD",   "BD/ YUS": "YU/BD",
    "YU -BD LINES": "YU/BD", "BLOOR DANFORTH & YONGE": "YU/BD",
    "ONGE-UNIVERSITY AND BL": "YU/BD",
    "YUS/ BD/ SHP": "YU/BD/SHP", "YU/BD/SHP": "YU/BD/SHP",
}
df["Line"] = df["Line"].replace(LINE_NORM)


VALID_LINES = {"YU", "BD", "SHP", "YU/BD", "YU/BD/SHP"}
length_before_drop = len(df)
df = df[df["Line"].isin(VALID_LINES)].copy()
print(f"Dropped {length_before_drop - len(df):,} rows with invalid 'Line' values")
print()
print(df["Line"].value_counts().to_string())



Rows dropped: 0
Dropped 0 rows with invalid 'Line' values

Line
YU           50734
BD           39498
SHP           3624
YU/BD         1495
YU/BD/SHP        3


In [10]:
# remove invalid stations
NON_PASSENGER_PATTERNS = [
    r"HILLCREST", r"CARHOUSE", r"\bYARD\b", r"SHOPS\b",
    r"BUILDING\b", r"COMPLEX\b", r"\bGATE\b", r"OFFICES\b",
    r"\bOPS\b", r"INGLIS", r"GO PROTOCOL", r"MCBRIEN",
    r"GUNN\b", r"SUBWAY CLOSURE", r"TRACK LEVEL ACTIVITY",
]
pattern = "|".join(NON_PASSENGER_PATTERNS)
mask = df["Station"].str.upper().str.contains(pattern, regex=True, na=False)
length_before_drop = len(df)
df = df[~mask].copy()

# fill remaining null line values based on mapping
station_line_map = (
    df.dropna(subset=["Line"])
    .groupby("Station")["Line"]
    .agg(lambda x: x.mode().iloc[0]) # most common Line for that station
)
still_null = df["Line"].isna().sum()
if still_null:
    df["Line"] = df["Line"].fillna(df["Station"].map(station_line_map))
    print(f"Filled {still_null} missing Line values from station mapping")
    df = df.dropna(subset=["Line"]).copy() # drop any that couldn't be resolved

print(f"Still_null: {still_null}")
print(f"Shape after cleaning : {df.shape[0]:,} rows")

Still_null: 0
Shape after cleaning : 94,626 rows


In [11]:
# we noticed that codes in the dataframe have typos, so we will correct them here
TYPO_MAP = {
    "MUNCA": "MUNOA",
    "TUNCA": "TUNOA",
    "TRNCA": "TRNOA",
}
df["Code"] = df["Code"].replace(TYPO_MAP)

# merge code descriptions
df_codes_raw = pd.read_excel(
    DATA_DIR / "ttc-subway-delay-codes.xlsx",
    sheet_name="Sheet 1", header=None
)

# the xlsx has an empty row at row index 1; data starts at row 2
# subway codes are in columns 2 (code) and 3 (description)
df_codes = df_codes_raw[[2, 3]].iloc[2:].copy()
df_codes.columns = ["Code", "Description"]
df_codes = df_codes.dropna(subset=["Code"]).reset_index(drop=True)
df_codes["Code"] = df_codes["Code"].astype(str).str.strip()
df_codes["Description"] = df_codes["Description"].astype(str).str.strip()

df = df.merge(df_codes, on="Code", how="left")
df["Description"] = df["Description"].fillna("Unknown")

matched_pct = (df["Description"] != "Unknown").mean()
print(f"Codes matched to description: {matched_pct:.1%}")

Codes matched to description: 99.6%


In [12]:
# assigning description categories for future visualization
def assign_category(desc):
    d = str(desc).lower()
    if any(w in d for w in ["injured", "ill customer", "passenger alarm", "suicide"]):
        return "Passenger / Medical"
    if any(w in d for w in ["debris", "unauthorized", "trespass", "police", "security", "assault"]):
        return "Security / Safety"
    if any(w in d for w in ["signal", "switch", "atc ", "power failure", "track ", "electrical"]):
        return "Track / Signal / Power"
    if any(w in d for w in ["door", "brake", "mechanical", "motor", "coupl", "air condition", "equipment"]):
        return "Equipment / Mechanical"
    if any(w in d for w in ["operator", "crew", "late leaving", "vision"]):
        return "Operations"
    if any(w in d for w in ["weather", "ice", "snow", "flood"]):
        return "Weather / Environmental"
    return "Other / Unknown"

df["Code_Category"] = df["Description"].apply(assign_category)
print()

print(df["Code_Category"].value_counts().to_string())


Code_Category
Other / Unknown            46834
Passenger / Medical        19553
Equipment / Mechanical      8442
Operations                  7479
Security / Safety           7189
Track / Signal / Power      4619
Weather / Environmental      510


In [13]:
# Date arrives as datetime64 from xlsx and as string from csv, so we normalise
df["Date_str"] = pd.to_datetime(df["Date"]).dt.strftime("%Y-%m-%d")
df["Time_str"] = df["Time"].astype(str).str.strip().str[:5]

df["Datetime"] = pd.to_datetime(
    df["Date_str"] + " " + df["Time_str"],
    format="%Y-%m-%d %H:%M",
    errors="coerce"
)
to_drop = df["Datetime"].isna().sum()
if to_drop:
    print(f"Dropping {to_drop} rows with unparseable datetime")
    df = df.dropna(subset=["Datetime"]).copy()

In [14]:
# create and extract time-based features for modelling
df["Year"]      = df["Datetime"].dt.year
df["Month"]     = df["Datetime"].dt.month
df["Day_Num"]   = df["Datetime"].dt.day
df["Hour"]      = df["Datetime"].dt.hour
df["Weekday"]   = df["Datetime"].dt.weekday # 0 = Monday, 6 = Sunday
df["DayName"]   = df["Datetime"].dt.day_name()
df["Peak_Hour"] = df["Hour"].apply(lambda x: 1 if (7 <= x <= 10) or (16 <= x <= 19) else 0)
df["Weekend"]   = (df["Weekday"] >= 5).astype(int)
df["Season"]    = df["Month"].map({
    12:"Winter", 1:"Winter", 2:"Winter",
    3:"Spring",  4:"Spring", 5:"Spring",
    6:"Summer",  7:"Summer", 8:"Summer",
    9:"Fall",   10:"Fall",  11:"Fall",
})

# drop helper columns
df = df.drop(columns=["Date_str", "Time_str"])

print(f"Final shape : {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Date range  : {df['Datetime'].min().date()} to {df['Datetime'].max().date()}")
print()
print("Min Delay distribution:")
print(df["Min Delay"].describe().round(2).to_string())
print()
# Bound column will not be used for modelling, but will be used for Power BI visualization, so we will keep it for now and just report the null percentage
print(f"Bound column: {df['Bound'].isna().mean():.1%} nulls")


Final shape : 94,626 rows x 22 columns
Date range  : 2022-01-01 to 2026-01-31

Min Delay distribution:
count    94626.00
mean         3.02
std         11.42
min          0.00
25%          0.00
50%          0.00
75%          4.00
max        900.00

Bound column: 34.1% nulls


In [16]:
df.describe()

,Min Delay,Min Gap,Vehicle,Datetime,Year,Month,Day_Num,Hour,Weekday,Peak_Hour,Weekend
count,94626.000000,94626.000000,94626.000000,94626,94626.000000,94626.000000,94626.000000,94626.000000,94626.000000,94626.000000,94626.000000
mean,3.016317,4.337835,3296.700864,2024-03-08 23:17:37.419314,2023.695168,6.394659,15.827204,13.011181,2.854469,0.389745,0.244742
min,0.000000,0.000000,0.000000,2022-01-01 01:02:00,2022.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,2023-03-27 18:48:30,2023.000000,3.000000,8.000000,8.000000,1.000000,0.000000,0.000000
50%,0.000000,0.000000,5136.000000,2024-04-03 18:04:30,2024.000000,6.000000,16.000000,14.000000,3.000000,0.000000,0.000000
75%,4.000000,8.000000,5616.000000,2025-02-27 09:17:45,2025.000000,10.000000,23.000000,18.000000,4.000000,1.000000,0.000000
max,900.000000,906.000000,9546.000000,2026-01-31 23:49:00,2026.000000,12.000000,31.000000,23.000000,6.000000,1.000000,1.000000
std,11.415984,11.377494,2733.986424,NaN,1.138903,3.543989,8.793232,6.583056,1.941313,0.487695,0.429937
